# Importamos las librerias

In [150]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sb

In [151]:
df_true = pd.read_csv("C:/Users/salaz/Downloads/Carpeta visual studio code/Carpeta visual studio code/Machine learning/regresion_Logistica/datasets/data1/True.csv")
df_false =pd.read_csv("C:/Users/salaz/Downloads/Carpeta visual studio code/Carpeta visual studio code/Machine learning/regresion_Logistica/datasets/data1/Fake.csv")

In [152]:
df_true.head(5)

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [153]:
df_false.head(5)

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


### VAMOS A CREAR UNA NUEVA COLUMNA PARA ETIQUETAR LAS NOTICIAS FALSAS CON LAS VERDADERAS PARA PRIORI UNIRLAS EN UN SOLO DATAFRAME

In [154]:
df_true['label'] = 'TRUE'
df_false['label'] = 'FALSE'

In [155]:
df_true.head(1)

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",TRUE


In [156]:
df_false.head(1)

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",FALSE


In [157]:
df = pd.concat([df_true , df_false], ignore_index=True)

In [158]:
df.tail(5)

,title,text,subject,date,label
44893,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",FALSE
44894,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",FALSE
44895,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",FALSE
44896,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",FALSE
44897,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",FALSE


### verificamos los resultados

In [159]:
print(f'Total de noticias verdaderas : {len(df_true)}')
print(f'Total de noticias falsas : {len(df_false)}')
print(f'Total de noticias {len(df)}')

Total de noticias verdaderas : 21417
Total de noticias falsas : 23481
Total de noticias 44898


# 2. Exploracion de los datos

In [160]:
# DISTRIBUCION DE NOTICIAS POR ETIQUETA
df['label'].value_counts()

label
FALSE    23481
TRUE     21417
Name: count, dtype: int64

In [161]:
# VEAMOS UN EJEMPLO DE UNA NOTICIA REAL
print("=" * 80)
print("Noticia Real")
print("=" * 80 )
print(f"\n titulo: {df[df['label'] == 'TRUE'].iloc[0]['title']}")
print(f'\n texto: {df[df['label'] == 'TRUE'].iloc[0]['text'][:500]}......')

Noticia Real

 titulo: As U.S. budget fight looms, Republicans flip their fiscal script

 texto: WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way among Republicans, U.S. Representative Mark Meadows, speaking on CBS’ “Face the Nation,” drew a hard line on federal spending, which lawmakers are bracing to do battle over in January. When they retur......


In [162]:
# VEAMOS UN EJEMPLO DE UNA NOTICIA FALSA
print("=" * 80)
print("NOTICIA FALSA")
print("=" * 80)
print(f'\n titulo: {df[df['label'] == 'FALSE'].iloc[0]['title']}')
print(f'\n texto : {df[df['label']== 'FALSE'].iloc[0]['text'][:500]} ....')

NOTICIA FALSA

 titulo:  Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing

 texto : Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a gr ....


# 3. PROCESAMIENTO DEL TEXTO

Aquí nos encontramos con uno de los **grandes retos** del Machine Learning aplicado a texto: los datos no vienen limpios. Las noticias pueden contener:

- **Código HTML** (etiquetas como `<p>`, `<br>`, `<div>`, etc.)
- **URLs** (enlaces a páginas web)
- **Signos de puntuación** que no aportan información útil
- **Palabras muy comunes** que aparecen en todas las noticias y no ayudan a distinguir entre falsas y verdaderas (por ejemplo: "the", "is", "and", "a"...)

Antes de poder entrenar nuestro modelo, necesitamos **procesar y limpiar el texto** para quedarnos únicamente con la información relevante. Vamos a hacerlo paso a paso.

#### 3.1. Eliminación de código HTML

Muchas noticias, especialmente las recopiladas de internet, pueden contener restos de código HTML. Por ejemplo, en lugar de ver un párrafo limpio, podemos encontrarnos algo como:

```
<p>This is a <b>breaking</b> news story...</p>
```

Necesitamos una forma de quedarnos solo con el texto y eliminar todas las etiquetas HTML.

**Pista**: Puedes utilizar la librería `BeautifulSoup` del paquete `bs4`, que nos permite parsear HTML fácilmente. La función `BeautifulSoup(texto, "html.parser").get_text()` devuelve únicamente el texto sin etiquetas.

In [163]:
%pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [164]:
    # IMPORTAMOS WARINGS PARA CONTROLAR LAS ADVERTENCIAS QUE MUESTRE PYTHON
    # IMPORTAMOS BS4 DE LA LIBRERIA BEAUTIFULSOUP PARA PODER LIMPIAR
import warnings
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

In [165]:
def strip_html(text):
    """ ELIMINA LAS ETIQUETAS HTML DE UN TEXTO"""
    return BeautifulSoup(text, 'html.parser').get_text()

In [166]:
print(strip_html(df.iloc[0]['text'])[:300])

WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 


#### 3.2. Eliminación de URLs

Las noticias también pueden contener URLs (direcciones web) que no aportan información útil para nuestro clasificador. Necesitamos eliminarlas.

**Pista**: Puedes utilizar **expresiones regulares** con la librería `re`. El patrón `https?://\S+` captura la mayoría de URLs (busca texto que empiece por `http://` o `https://` seguido de cualquier carácter que no sea un espacio).

In [167]:
import re

In [168]:
def remove_urls(text):
    """Elimina las urls del texto"""
    return re.sub(r'https?://\s+', '', text)

#### 3.3. Eliminación de signos de puntuación y conversión a minúsculas

Los signos de puntuación (comas, puntos, signos de exclamación, etc.) generalmente no aportan información útil para distinguir entre noticias falsas y verdaderas. Además, queremos que nuestro algoritmo trate igual las palabras independientemente de si están en mayúsculas o minúsculas (por ejemplo, "President" y "president" deberían ser la misma palabra).

**Pista**: Utiliza el método `.lower()` de Python para convertir a minúsculas y la librería `string` para obtener una lista de todos los signos de puntuación.

In [169]:
import string
#Estos son todos los signos de puntuacion que python reconoce
print(string.punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [170]:
def remove_punctuation(text):
    # convierte todo el texto que contenga mayuscula a minuscula
    text = text.lower()
    #Eliminmaos la puntuacion estandar ASSCII
    text = text.translate(str.maketrans('','', string.punctuation))
    #Eliminamos tambien las comillas tipograficas y otros caraceteres especiales
    # que no estan incluidos en el string.remove_punctuation
    text= re.sub(r'[\u2018\u2019\u201C\u201D\u2013\u2014\u2026]', '', text)
    return text


In [171]:
# EJEMPLO
ejemplo = "BREAKING NEWS! The president said: 'no comment' ."
print("original: ", ejemplo)
print("Procesado: ", remove_punctuation(ejemplo))

original:  BREAKING NEWS! The president said: 'no comment' .
Procesado:  breaking news the president said no comment 


#### 3.4. Eliminación de stopwords y Stemming

Ahora vamos a aplicar dos técnicas fundamentales del **Procesamiento de Lenguaje Natural (NLP)** que veremos con más profundidad:

**Stopwords (palabras vacías)**: Son palabras muy comunes en un idioma que aparecen en prácticamente todos los textos y, por lo tanto, no aportan información útil para distinguir entre categorías. Algunos ejemplos en inglés son: *"the"*, *"is"*, *"and"*, *"a"*, *"in"*, *"to"*...

Imagina que estás intentando distinguir si una noticia es falsa o verdadera. La palabra *"the"* va a aparecer en ambos tipos de noticias con una frecuencia similar, así que no nos ayuda. Sin embargo, palabras como *"hoax"* (engaño) o *"verified"* (verificado) podrían ser más informativas.

**Stemming (lematización)**: Es el proceso de reducir una palabra a su **raíz**. Por ejemplo:
- *"running"*, *"runs"*, *"ran"* → **"run"**
- *"playing"*, *"played"*, *"plays"* → **"play"**

¿Por qué nos interesa esto? Porque para nuestro algoritmo, *"running"* y *"runs"* son palabras completamente diferentes, cuando en realidad tienen el mismo significado base. Al reducirlas a su raíz, agrupamos las variantes de una misma palabra.

**Pista**: Revisa la clase `PorterStemmer()` de nltk y el listado de stopwords en `nltk.corpus.stopwords.words('english')`. Para dividir un texto en palabras individuales, utiliza `nltk.tokenize.word_tokenize()`.

In [172]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [173]:
import nltk
# DESCARGAMOS LOS RECURSOS NECESARIOS DE NLTK (solo la primera vez)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet = True)

True

In [174]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

In [175]:
# ALGUNAS STOPWORDS EN INGLES Y PALABRAS DE SESGO DE FUENTE
base_stopwords = set(stopwords.words('english'))
stop_words = base_stopwords.union({
    'reuters', 'said', 'say', 'via', 'video', 'image', 'pictwittercom', 
    'watch', 'featured', 'monday', 'tuesday', 'wednesday', 'thursday', 
    'friday', 'saturday', 'sunday'
})
print(f'Numero de StopWords (incluyendo palabras de sesgo): {len(stop_words)}')
print(f'\n Algunos ejemplos: {list(stop_words)[:20]}')

Numero de StopWords (incluyendo palabras de sesgo): 214

 Algunos ejemplos: ['should', "they're", 'his', 'shouldn', 'reuters', 'said', 'wednesday', 'friday']


In [176]:
#VEAMOS COMO FUNCIONA EL STEMMER
stemmer=PorterStemmer()
palabras = ["running", "runs", "ran", "playing", "played", "investigation", "investigating"]
for palabra in palabras:
    print(f'{palabra:20s} -> {stemmer.stem(palabra)}')

running              -> run
runs                 -> run
ran                  -> ran
playing              -> play
played               -> play
investigation        -> investig
investigating        -> investig


In [177]:
#vamos a mostrar unos ejemplos de todo el proceso completo de tokenizar
#eliminar stopwords, y aplicar steamming
ejemplo = "The president was running as investigation into the reported claims"
print(f'texto original: {ejemplo}')
#1. TOKENIZAR (DIVIDIR LAS PALABRAS)
tokens = word_tokenize(ejemplo.lower())
# 2. Eliminar stopwords (eliminar los adverbios, adjetivos, etc)
tokens_filtrados =[t for t in tokens if t not in stop_words]
print("\nSin Stopwords: ", tokens_filtrados)
# 3. Aplicar stemming (volver las palabras a su estado raiz)
tokens_stem=[stemmer.stem(t) for t in tokens_filtrados]
print("\nCon Stemming: ", tokens_stem)

texto original: The president was running as investigation into the reported claims

Sin Stopwords:  ['president', 'running', 'investigation', 'reported', 'claims']

Con Stemming:  ['presid', 'run', 'investig', 'report', 'claim']


### 4. Función de preprocesamiento completa

Ahora que hemos implementado todas las transformaciones de texto por separado, vamos a ponerlas todas en común dentro de una única función que procese una noticia de principio a fin.

In [178]:
def preprocesar_texto(text):
    """
    Aplicar todas las transformaciones sobre un texto:
    0. Eliminar prefijos de fuentes
    1. Eliminar etiquetas HTML
    2. Eliminar las URLs
    3. Converir a minusculas
    4.Eliminar signos de puntuacion
    5. Tokenizar
    6. Eliminar Stopwords
    7. Aplicar Stemming
    retornar el texto preprocesado como un unico string
    """
    #Eliminar los prefijos de fuentes
    text = re.sub(r'^[A-Z\s,.]+ \([^)]+\)\s*[-–—]?\s*', '', text)

    #Eliminar HTML
    text= strip_html(text)

    #Eliminar las URLs
    text = remove_urls(text)

    #Minusculas y puntuacion
    text = remove_punctuation(text)

    # tokenizar
    tokens = word_tokenize(text)

    #stemming
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(t) for t in tokens]
    
    # Unimos los tokens de nuevo en un string
    return " " .join(tokens)

In [179]:
texto_original = df.iloc[0]["text"]
print("TEXTO ORIGINAL (primero 300 caracteres): ")
print(texto_original[:300])
print("\n" + "=" * 80)
print("\n Texto procesado (primeros 300 caracteres)")
print(preprocesar_texto(texto_original)[:300])


TEXTO ORIGINAL (primero 300 caracteres): 
WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 


 Texto procesado (primeros 300 caracteres)
the head of a conserv republican faction in the us congress who vote thi month for a huge expans of the nation debt to pay for tax cut call himself a fiscal conserv on sunday and urg budget restraint in 2018 in keep with a sharp pivot under way among republican us repres mark meadow speak on cb face


### 5. Aplicar el preprocesamiento al conjunto de datos

Ahora que tenemos nuestra función de preprocesamiento, vamos a aplicarla sobre todas las noticias del conjunto de datos. Este proceso puede tardar un poco ya que tenemos más de 44.000 noticias.

**Pista**: Utiliza el método `.apply()` de Pandas para aplicar la función de preprocesamiento sobre la columna `"text"` del DataFrame. Para empezar, trabaja con un subconjunto usando `.sample()` o seleccionando las primeras N filas.

In [180]:
# Mezclamos el dataFrame para tener noticias falsas y verdaderas tomando en cuenta usamos
# el metodo concat y solo une dos dataframes pero no las entrevera
df_sample = df.sample(n=1000, random_state=42)
print(f'Tamaño del dataFrame mezclado (subconjunto) : {len(df_sample)}')
print("Distribucion de las Etiquetas")
print(df_sample['label'].value_counts())

Tamaño del dataFrame mezclado (subconjunto) : 1000
Distribucion de las Etiquetas
label
FALSE    536
TRUE     464
Name: count, dtype: int64


In [181]:
#Aplicamos el preprocesamiento a todas las noticas del subconjunto
print("Preprosesando noticias.....")
df_sample['text_clean'] = df_sample['text'].apply(preprocesar_texto)
print("LISTO")

Preprosesando noticias.....
LISTO


In [182]:
#vemos el resultado
df_sample.head(10)

,title,text,subject,date,label,text_clean
22216,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",FALSE,donald trump s white hous is in chao and they ...
27917,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",FALSE,now that donald trump is the presumpt gop nomi...
25007,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",FALSE,mike penc is a huge homophob he support exgay ...
1377,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",TRUE,california attorney gener xavier becerra said ...
32476,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",FALSE,twist reason is all that come from pelosi thes...
5766,"As private lawyer, Trump high court pick was f...",WASHINGTON (Reuters) - As a lawyer in private ...,politicsNews,"February 1, 2017",TRUE,as a lawyer in privat practic for a decad pres...
16230,Yemeni Salafist imam killed in Aden: sources,ADEN (Reuters) - A Salafist imam was shot dead...,worldnews,"October 28, 2017",TRUE,a salafist imam wa shot dead by gunmen earli o...
20911,FBI says witnesses in U.S. probe into Malaysia...,KUALA LUMPUR (Reuters) - Potential witnesses t...,worldnews,"September 6, 2017",TRUE,potenti wit to the multibillion dollar scandal...
35838,An Easy To Read Chart Shows How Bernie Sanders...,The goal of socialism is communism. -Vladimi...,politics,"Feb 24, 2016",FALSE,the goal of social is commun vladimir lenin co...
33155,MMA FIGHTER JAKE SHIELDS Embarrasses Cowards I...,Opposing views and beliefs has much of this co...,politics,"Feb 4, 2017",FALSE,oppos view and belief ha much of thi countri i...


In [183]:
#Ejemplo del antes y despues
print('ANTES')
print(df_sample.iloc[0]['text'][:200])
print('Despues')
print(df_sample.iloc[0]['text_clean'][:200])

ANTES
Donald Trump s White House is in chaos, and they are trying to cover it up. Their Russia problems are mounting by the hour, and they refuse to acknowledge that there are problems surrounding all of th
Despues
donald trump s white hous is in chao and they are tri to cover it up their russia problem are mount by the hour and they refus to acknowledg that there are problem surround all of thi to them it s fak


### 6. Codificando el texto: vectorización

Este es uno de los pasos más **importantes** y que puede resultar más novedoso. Hasta ahora, los conjuntos de datos con los que hemos trabajado en el curso tenían columnas numéricas (precio, número de equipos, etc.). Sin embargo, en este caso nuestros datos son **texto** y los algoritmos de Machine Learning necesitan recibir **números**.

¿Cómo podemos convertir texto en números? A este proceso se le llama **vectorización** y existen varias técnicas. Nosotros vamos a utilizar una de las más sencillas e intuitivas: **Bag of Words (Bolsa de Palabras)**.

#### ¿Qué es Bag of Words?

La idea es muy sencilla. Imagina que tenemos solo dos noticias en nuestro conjunto de datos:

- **Noticia 1**: *"president signs new law today"*
- **Noticia 2**: *"new study shows results today"*

Lo que hace Bag of Words es:

1. **Crear un vocabulario** con todas las palabras únicas que aparecen en el conjunto de datos: `["law", "new", "president", "results", "shows", "signs", "study", "today"]`

2. **Representar cada noticia como un vector** de números, donde cada número indica **cuántas veces aparece cada palabra** del vocabulario en esa noticia:

| | law | new | president | results | shows | signs | study | today |
|---|---|---|---|---|---|---|---|---|
| **Noticia 1** | 1 | 1 | 1 | 0 | 0 | 1 | 0 | 1 |
| **Noticia 2** | 0 | 1 | 0 | 1 | 1 | 0 | 1 | 1 |

De esta manera, cada noticia se convierte en un **vector numérico** que el algoritmo de Machine Learning puede entender. A este proceso se le llama **vectorización** y es fundamental cuando trabajamos con texto.

En la siguiente sección del curso profundizaremos mucho más en estas técnicas. De momento, vamos a utilizar la implementación que nos proporciona Sklearn.

**Pista**: Revisa la clase `CountVectorizer` de Sklearn. Esta clase implementa exactamente el proceso de Bag of Words que hemos descrito. Tiene dos métodos principales:
- `.fit()`: aprende el vocabulario a partir de los textos de entrenamiento.
- `.transform()`: transforma los textos en vectores numéricos usando el vocabulario aprendido.

In [184]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [185]:
#primero un ejemplo de como funciona 
ejemplos_texto=[
    'president signs new law today',
    'new study shows results today'
]

vectorizer_ejemplo = CountVectorizer()
vectorizer_ejemplo.fit(ejemplos_texto)

print('vocabulario aprendido')
print(vectorizer_ejemplo.get_feature_names_out())

vocabulario aprendido
['law' 'new' 'president' 'results' 'shows' 'signs' 'study' 'today']


In [186]:
# Transformamos los textos en vectores
vectores_ejemplo= vectorizer_ejemplo.transform(ejemplos_texto)

#visualizamos la matriz resultante
pd.DataFrame(
    vectores_ejemplo.toarray(),
    columns=vectorizer_ejemplo.get_feature_names_out(),
    index=['Noticia 1', 'Noticia 2']
)

,law,new,president,results,shows,signs,study,today
Noticia 1,1,1,1,0,0,1,0,1
Noticia 2,0,1,0,1,1,0,1,1


Como puedes ver, el resultado es exactamente la tabla que hemos descrito antes. Cada fila representa una noticia y cada columna representa una palabra del vocabulario. Los valores indican cuántas veces aparece cada palabra en cada noticia.

Ahora vamos a aplicar este mismo proceso sobre nuestro conjunto de datos real.

In [187]:
# Aplicamos CountVectorizer sobre nuestros textos procesados
vectorizer = CountVectorizer()
vectorizer.fit(df_sample['text_clean'])
print(f'tamaño del vocabulario {len(vectorizer.get_feature_names_out())} palabras unicas')

tamaño del vocabulario 19400 palabras unicas


In [188]:
# Transformamos los testos a vectores
X_vect= vectorizer.transform(df_sample['text_clean'])
print(f'Dimensiones de la matriz resultante: {X_vect.shape}')
print(f'  -> {X_vect.shape[0]} noticias')
print(f'  -> {X_vect.shape[1]} palabras en el vocabulario')

Dimensiones de la matriz resultante: (1000, 19400)
  -> 1000 noticias
  -> 19400 palabras en el vocabulario


In [189]:
# veamos las primeras filas de la matriz (solo las primeras 10 columnas)
pd.DataFrame(
    X_vect.toarray(),
    columns=vectorizer.get_feature_names_out()
)

,000004,00009,0006,005380k,0100,015760k,020,0259,0300,0307,...,zip,zipper,zippi,zolani,zombi,zone,zoran,zuckerberg,zuma,zweli
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**Nota**: La matriz resultante es muy grande (cada noticia se representa con un vector de miles de elementos) y la mayoría de los valores son 0 (una noticia individual solo contiene una pequeña fracción de todas las palabras del vocabulario). Este tipo de matrices se llaman **matrices dispersas (sparse)** y Sklearn las maneja de forma eficiente internamente.

### 7. Entrenamiento del algoritmo

Ahora vamos a entrenar un modelo de **Regresión Logística** que aprenderá de estos vectores a clasificar las noticias en falsas y verdaderas. El proceso es exactamente igual que el que hemos visto con la Regresión Lineal: le proporcionamos datos de entrenamiento (noticias con sus etiquetas) y el algoritmo aprende los patrones que distinguen una noticia falsa de una verdadera.

**Pista 1**: Comienza utilizando un número reducido de noticias para que no tarde demasiado tiempo, por ejemplo 1000 noticias. Aplica todas las transformaciones que hemos implementado anteriormente sobre ellas.

In [190]:
# leemos unicamente  un subconjuntos de 1500 noticias
df_all = df.sample(n=1500, random_state=42)

#nos quedamos con 1000 noticias para el entrenamiento
df_sample = df_all.iloc[:1000]
print(f'tamaño del subconjunto : {len(df_sample)}')
print(f'Distribucion de las etiquetas: ')
print(df_sample['label'].value_counts())

tamaño del subconjunto : 1000
Distribucion de las etiquetas: 
label
FALSE    536
TRUE     464
Name: count, dtype: int64


In [191]:
#preprocesamos el texto
print('preprocesando noticias......')
df_sample['text_clean'] = df_sample['text'].apply(preprocesar_texto)
print('!Listo!')

preprocesando noticias......
!Listo!


**Pista 2:** Aplica la vectorización al conjunto de datos para representar las noticias de manera numérica.

In [192]:
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(df_sample['text_clean'])

In [193]:
print(X_train.toarray())
print(f'Features : {len(vectorizer.get_feature_names_out())}')

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
Features : 19400


In [194]:
pd.DataFrame(X_train.toarray(), columns=[vectorizer.get_feature_names_out()])

,000004,00009,0006,005380k,0100,015760k,020,0259,0300,0307,...,zip,zipper,zippi,zolani,zombi,zone,zoran,zuckerberg,zuma,zweli
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [195]:
y_train = df_sample['label']
y_train

22216    FALSE
27917    FALSE
25007    FALSE
1377      TRUE
32476    FALSE
         ...  
27000    FALSE
25165    FALSE
20813     TRUE
11756     TRUE
33887    FALSE
Name: label, Length: 1000, dtype: str

**Pista 3**: Entrena el algoritmo `LogisticRegression` de Sklearn. Este algoritmo se basa en aprendizaje supervisado.

In [196]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter = 1000)
clf.fit(X_train , y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

### 8. Predicción

Ya tenemos nuestro algoritmo entrenado y listo para realizar predicciones, lo único que nos queda es probar qué tal se comporta para noticias que no ha visto nunca.

**Pista**: Revisa el método `predict()` que tiene la clase `LogisticRegression`. Ten en cuenta que cuando recibas una nueva noticia para la que quieres realizar una predicción debes aplicar sobre ella todas las transformaciones que hemos realizado anteriormente. **Al aplicar la vectorización utiliza únicamente el método `transform()` del objeto `CountVectorizer`**.

##### Lectura de un conjunto de noticias nuevas

In [197]:
# Tomamos las 500 noticias que no se han utilizado para entrenar el algoritmo
df_test = df_all.iloc[1000:]
print('preprocesando noticias del test....')
df_test['text_clean'] = df_test['text'].apply(preprocesar_texto)
print('!LISTO!')

preprocesando noticias del test....
!LISTO!


<!-- ### 7. Entrenamiento del algoritmo -->

In [198]:
X_test = df_test['text_clean']
y_test = df_test['label']
print(f'noticias del test: {len(X_test)}')

noticias del test: 500


##### Preprocesamiento de las noticias con el vectorizador creado anteriormente

In [199]:
#Aplicamos CountVectorizer (solo .transform(), No .fit())
X_test = vectorizer.transform(X_test)

##### Predicción del tipo de noticia

In [200]:
y_pred = clf.predict(X_test)
y_pred

array(['TRUE', 'TRUE', 'FALSE', 'FALSE', 'TRUE', 'FALSE', 'FALSE',
       'FALSE', 'FALSE', 'FALSE', 'TRUE', 'TRUE', 'FALSE', 'FALSE',
       'FALSE', 'TRUE', 'TRUE', 'TRUE', 'TRUE', 'FALSE', 'TRUE', 'TRUE',
       'TRUE', 'FALSE', 'TRUE', 'TRUE', 'FALSE', 'FALSE', 'FALSE',
       'FALSE', 'TRUE', 'FALSE', 'FALSE', 'FALSE', 'FALSE', 'TRUE',
       'TRUE', 'TRUE', 'TRUE', 'FALSE', 'FALSE', 'FALSE', 'FALSE', 'TRUE',
       'FALSE', 'FALSE', 'FALSE', 'FALSE', 'TRUE', 'FALSE', 'TRUE',
       'FALSE', 'FALSE', 'TRUE', 'FALSE', 'TRUE', 'FALSE', 'FALSE',
       'FALSE', 'TRUE', 'FALSE', 'FALSE', 'TRUE', 'FALSE', 'FALSE',
       'FALSE', 'TRUE', 'TRUE', 'FALSE', 'TRUE', 'FALSE', 'TRUE', 'TRUE',
       'FALSE', 'FALSE', 'TRUE', 'TRUE', 'FALSE', 'TRUE', 'TRUE', 'FALSE',
       'FALSE', 'TRUE', 'TRUE', 'TRUE', 'FALSE', 'TRUE', 'FALSE', 'FALSE',
       'TRUE', 'FALSE', 'FALSE', 'FALSE', 'TRUE', 'FALSE', 'FALSE',
       'FALSE', 'FALSE', 'FALSE', 'TRUE', 'FALSE', 'TRUE', 'TRUE',
       'FALSE', 'FA

##### Evaluación de los resultados

In [201]:
from sklearn.metrics import accuracy_score
print('Accuaracy: {:.3f}'.format(accuracy_score(y_test, y_pred)))

Accuaracy: 0.952


### 9. Aumentando el conjunto de datos

Nuestro detector de noticias falsas ya funciona bastante bien con solo 1000 noticias. Sin embargo, los algoritmos de Machine Learning suelen funcionar **mejor** cuantos más datos de entrenamiento tienen. Prueba a entrenar el algoritmo con un conjunto de datos de entrenamiento más grande.

In [202]:
#Leemos  4200 noticias
df_grande = df.sample(n=4200 , random_state=42)
print('Preprocesando noticias....')
df_grande['text_clean'] = df_grande['text'].apply(preprocesar_texto)
print('Listo')

Preprocesando noticias....
Listo


In [203]:
# Separamos adecuadamente los datos en 80% entrenamiento y 20% prueba usando train_test_split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_grande['text_clean'], df_grande['label'], test_size=0.2, random_state=42, stratify=df_grande['label']
)
print(f'Noticias de entrenamiento: {len(X_train_raw)}')
print(f'Noticias de prueba: {len(X_test_raw)}')

Noticias de entrenamiento: 3360
Noticias de prueba: 840


In [204]:
# Usamos TfidfVectorizer limitando a 5000 características para evitar el sobreajuste
vectorizer = TfidfVectorizer(max_features=5000, min_df=3)
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)
print(f'Dimensiones de la matriz de entrenamiento: {X_train.shape}')

Dimensiones de la matriz de entrenamiento: (3360, 5000)


In [205]:
# Entrenamos el modelo de Regresión Logística con regularización moderada (C=0.5)
clf = LogisticRegression(C=0.5, max_iter=1000)
clf.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [208]:
# Realizamos predicciones en el conjunto de prueba
y_pred = clf.predict(X_test)

In [210]:
# Evaluamos la precisión en Entrenamiento y en Prueba para verificar que no haya sobreajuste
train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, y_pred)
print(f'Precisión en Entrenamiento (Train Accuracy): {train_acc:.3f}')
print(f'Precisión en Prueba (Test Accuracy): {test_acc:.3f}')

Precisión en Entrenamiento (Train Accuracy): 0.957
Precisión en Prueba (Test Accuracy): 0.941


In [217]:
# Reporte detallado de clasificación
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       FALSE       0.95      0.94      0.94       440
        TRUE       0.93      0.95      0.94       400

    accuracy                           0.94       840
   macro avg       0.94      0.94      0.94       840
weighted avg       0.94      0.94      0.94       840


### 10. Probando con noticias nuevas

Para terminar, vamos a probar nuestro modelo con noticias que escribamos nosotros mismos. De esta manera podemos ver cómo se comporta el modelo en un escenario más realista.

**Pista**: Al recibir una nueva noticia para clasificar, debes:
1. Aplicar la función `preprocesar_texto()` sobre ella.
2. Aplicar **únicamente** el método `.transform()` del `CountVectorizer` (NO `.fit()`).
3. Usar el método `.predict()` del modelo entrenado.
4. Opcionalmente, puedes usar `.predict_proba()` para ver las probabilidades que asigna el modelo a cada clase.

In [228]:
noticias_nuevas = [
    """El endeudamiento en los hogares
Lo mismo sostiene Mosquera: la decisión de un hogar de tomar o no un crédito depende del destino de esos recursos y de la forma en que planea pagarlos.

El decano de la UDLA plantea que ante ante El Niño y los apagones, la decisión de endeudarse corresponde a cada tomador de fondos, quien debe evaluar si puede cubrir el crédito a lo largo del tiempo. La situación ideal, dice, es tomar recursos para un negocio cuyo flujo de caja permita pagar el costo financiero de esa deuda.

En el crédito de consumo, la clave está en ver si las condiciones financieras de los hogares mejorarán en el futuro. Sin esa perspectiva, “tal vez estamos incurriendo en un problema de sobreendeudamiento”.
Este contenido ha sido publicado originalmente por EL COMERCIO. Si vas a hacer uso del mismo, por favor, cita la fuente y haz un enlace hacia la nota original en la dirección: https://www.elcomercio.com/actualidad/negocios/ahorrar-invertir-100-dolares-mes-convertirse-fortuna-interes-compuesto/"""
]

In [230]:
#preprocesamos las noticias
noticias_procesadas = [preprocesar_texto(n) for n in noticias_nuevas]

#vectorizamos usando el mismo vectorizer entrenado (solo .transform())
noticias_vect =vectorizer.transform(noticias_procesadas)

#Realizamos las predicciones
predicciones = clf.predict(noticias_vect)
probabilidades = clf.predict_proba(noticias_vect)
#realizamos las predicciones
for noticias , pred, prob in zip(noticias_nuevas , predicciones, probabilidades):
    print(f'Noticia: {noticias[:80]}...')
    print(f'  -> Prediccion: {pred}')
    print(f'  -> Probabilidad FAKE: {prob[0]:.2f} | Probabilidad REAL: {prob[1]:.2%}")')

Noticia: El endeudamiento en los hogares
Lo mismo sostiene Mosquera: la decisión de un ho...
  -> Prediccion: FALSE
  -> Probabilidad FAKE: 0.91 | Probabilidad REAL: 9.09%")
